The data for this tutorial can be downloaded at: https://drive.google.com/drive/folders/1z5Y2IldWLyiwSaRo2_7cEE3uvAbMZ6dj?usp=sharing

In [1]:
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import wandb  # Integrated for experiment tracking
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.autograd import Function

# --- 1. Dataset Class ---
class MNISTPickleDataset(Dataset):
    def __init__(self, file_path, mode="Train"):
        with open(file_path, 'rb') as f:
            data = pickle.load(f)

        key_map = {"Train": ("train", "train_labels"), 
                   "Val": ("val", "val_labels"), 
                   "Test": ("test", "test_labels")}
        
        img_key, lbl_key = key_map.get(mode, key_map["Test"])
        self.images = data[img_key]
        self.labels = data[lbl_key]

        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        label = int(self.labels[idx])
        if len(img.shape) == 2:
            img = np.stack([img] * 3, axis=-1)
        return self.transform(img), label

# --- 2. Model Components ---
class GradientReversal(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class DANN(nn.Module):
    def __init__(self):
        super(DANN, self).__init__()
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=5), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 50, kernel_size=5), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.label_predictor = nn.Sequential(
            nn.Linear(50 * 4 * 4, 100), nn.ReLU(), nn.Linear(100, 10)
        )
        self.domain_classifier = nn.Sequential(
            nn.Linear(50 * 4 * 4, 100), nn.ReLU(), nn.Linear(100, 2)
        )

    def forward(self, x, alpha=1.0):
        features = self.feature_extractor(x).view(-1, 50 * 4 * 4)
        reverse_features = GradientReversal.apply(features, alpha)
        return self.label_predictor(features), self.domain_classifier(reverse_features)

# --- 3. Evaluation Logic ---
def validate(model, loader, device, criterion):
    model.eval()
    loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            class_out, _ = model(imgs, alpha=0)
            loss += criterion(class_out, labels).item()
            _, pred = torch.max(class_out, 1)
            correct += (pred == labels).sum().item()
            total += labels.size(0)
    return loss / len(loader), 100 * correct / total

# --- 4. Main Training & WandB Logging ---
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
    
    # Initialize WandB
    wandb.init(
        project="dann-mnist-m",
        config={
            "learning_rate": 0.01,
            "architecture": "DANN",
            "dataset": "MNIST to MNIST-M",
            "epochs": 20,
            "batch_size": 64
        }
    )

    # Paths
    mnist_path = "/Users/robertosouza/Library/CloudStorage/OneDrive-UniversityofCalgary/Documents/Teaching/github/ENSF617-ENEN-645-W2026/Data/DANN/mnist_data.pkl"
    mnist_m_path = "/Users/robertosouza/Library/CloudStorage/OneDrive-UniversityofCalgary/Documents/Teaching/github/ENSF617-ENEN-645-W2026/Data/DANN/mnistm_data.pkl"

    # Loaders
    src_train = DataLoader(MNISTPickleDataset(mnist_path, "Train"), batch_size=64, shuffle=True, drop_last=True)
    tgt_train = DataLoader(MNISTPickleDataset(mnist_m_path, "Train"), batch_size=64, shuffle=True, drop_last=True)
    tgt_val = DataLoader(MNISTPickleDataset(mnist_m_path, "Val"), batch_size=64, shuffle=False)
    tgt_test = DataLoader(MNISTPickleDataset(mnist_m_path, "Test"), batch_size=64, shuffle=False)

    model = DANN().to(device)
    wandb.watch(model, log_freq=100) # Tracks gradients in wandb
    
    optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    criterion = nn.CrossEntropyLoss()

    epochs = 20
    for epoch in range(epochs):
        model.train()
        len_loader = min(len(src_train), len(tgt_train))
        
        for i, ((s_img, s_lbl), (t_img, _)) in enumerate(zip(src_train, tgt_train)):
            p = float(i + epoch * len_loader) / (epochs * len_loader)
            alpha = 2. / (1. + np.exp(-10 * p)) - 1

            optimizer.zero_grad()
            s_img, s_lbl, t_img = s_img.to(device), s_lbl.to(device), t_img.to(device)
            
            # Forward Source
            s_cls, s_dom = model(s_img, alpha)
            loss_s_cls = criterion(s_cls, s_lbl)
            loss_s_dom = criterion(s_dom, torch.zeros(s_img.size(0), dtype=torch.long).to(device))

            # Forward Target
            _, t_dom = model(t_img, alpha)
            loss_t_dom = criterion(t_dom, torch.ones(t_img.size(0), dtype=torch.long).to(device))

            total_loss = loss_s_cls + loss_s_dom + loss_t_dom
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

            # Log step-level metrics
            if i % 10 == 0:
                wandb.log({
                    "batch_total_loss": total_loss.item(),
                    "batch_class_loss": loss_s_cls.item(),
                    "batch_domain_loss": (loss_s_dom + loss_t_dom).item(),
                    "alpha": alpha
                })

        # Epoch Validation
        v_loss, v_acc = validate(model, tgt_val, device, criterion)
        wandb.log({
            "epoch": epoch + 1,
            "val_loss": v_loss,
            "val_accuracy": v_acc
        })
        
        print(f"Epoch {epoch+1:02d} | Val Loss: {v_loss:.3f} | Target Acc: {v_acc:.2f}%")

    # Final Test Set Evaluation
    test_loss, test_acc = validate(model, tgt_test, device, criterion)
    wandb.log({"test_loss": test_loss, "test_accuracy": test_acc})
    print(f"\n--- Final Test Accuracy: {test_acc:.2f}% ---")
    
    wandb.finish()

if __name__ == "__main__":
    main()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/robertosouza/.netrc.
wandb: Currently logged in as: roberto-medeirosdeso (ai2lab_u_calgary) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/var/folders/j4/_hc8rl9x3t102k8g5bns65d80000gn/T/ipykernel_41192/2317828550.py:16: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  data = pickle.load(f)


Epoch 01 | Val Loss: 1.415 | Target Acc: 60.28%
Epoch 02 | Val Loss: 1.589 | Target Acc: 64.26%
Epoch 03 | Val Loss: 1.621 | Target Acc: 62.52%
Epoch 04 | Val Loss: 1.280 | Target Acc: 68.63%
Epoch 05 | Val Loss: 1.272 | Target Acc: 66.85%
Epoch 06 | Val Loss: 1.061 | Target Acc: 71.95%
Epoch 07 | Val Loss: 0.933 | Target Acc: 73.22%
Epoch 08 | Val Loss: 1.025 | Target Acc: 74.12%
Epoch 09 | Val Loss: 1.186 | Target Acc: 69.75%
Epoch 10 | Val Loss: 0.885 | Target Acc: 72.58%
Epoch 11 | Val Loss: 0.951 | Target Acc: 73.38%
Epoch 12 | Val Loss: 1.056 | Target Acc: 71.82%
Epoch 13 | Val Loss: 0.989 | Target Acc: 72.09%
Epoch 14 | Val Loss: 1.159 | Target Acc: 69.41%
Epoch 15 | Val Loss: 0.885 | Target Acc: 74.29%
Epoch 16 | Val Loss: 1.008 | Target Acc: 72.78%
Epoch 17 | Val Loss: 0.896 | Target Acc: 71.91%
Epoch 18 | Val Loss: 0.883 | Target Acc: 75.75%
Epoch 19 | Val Loss: 0.889 | Target Acc: 73.47%
Epoch 20 | Val Loss: 0.933 | Target Acc: 72.62%


wandb: ERROR The nbformat package was not found. It is required to save notebook history.



--- Final Test Accuracy: 73.13% ---


alpha,▁▃▃▄▅▅▆▇▇▇▇█████████████████████████████
batch_class_loss,▄▃█▄▂▆▂▂▃▄▁▁▅▂▂▃▂▃▂▄▂▁▁▂▂▃▁▁▁▇▁▁▁▂▁▁▁▁▁▃
batch_domain_loss,▁▂▃▅▅▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▆█▇▇▇▇▇▇▇▇██▇▇▇█▇
batch_total_loss,▂▃▃▁█▅▅▅▇▇▆▆▆▆▆▆▆▆▆▇▆▇▆▆▆▆▆▇▇▆▆▇▇▆▇█▇▇▆▇
epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
test_accuracy,▁
test_loss,▁
val_accuracy,▁▃▂▅▄▆▇▇▅▇▇▆▆▅▇▇▆█▇▇
val_loss,▆██▅▅▃▁▂▄▁▂▃▂▄▁▂▁▁▁▁
alpha,0.99991
batch_class_loss,0.00204
